# Applied Data Science (STAT GR 5243) Project 3

## Prediction on Mortgage Loan Application 

**Team 23:** Veronica Joe (jj3470), Crystal Guo (lg3434), Shuxuan Xu (sx2412), Fangyi Lin (fl2748)

**Github Repository:** https://github.com/xsxsx-999/5243-project4

**Mortgage Approval Estimator ShinyApp:** https://crystalguo.shinyapps.io/mortgage-approval-estimator/




## 1. Project Overview

### 1.1 Background and Objective

The objective of this project is to build an end-to-end machine learning pipeline to analyze mortgage loan decisions within the United States mortgage market. Using data from the Home Mortgage Disclosure Act (HMDA), this study focuses on applications with definitive outcomes and models whether an application is originated or denied.

By engineering robust features and training predictive models, this project seeks to identify the key drivers—ranging from applicant demographics to loan characteristics—that influence mortgage origination outcomes. The final goal is not only to predict application outcomes, but also to explain the main factors behind model decisions through feature-signal analysis, model interpretation, and fairness-oriented diagnostics.

### 1.2 Data Collection

To construct a focused and computationally viable dataset, our data acquisition and sampling strategy followed a strict, predefined methodology:

* **Data Source & Geographical Scope:** We obtained historical mortgage application records from the HMDA database through the FFIEC/CFPB HMDA Data Browser API. The dataset spans a six-year period from **2019 to 2024**. To capture large and economically important mortgage markets while managing data volume, the geographical scope was restricted to **California (CA) and Texas (TX)**, two of the largest U.S. housing markets by housing-unit count (FFIEC, 2025).

* **Target Formulation & Filtering:** The full HMDA dataset contains multiple stages of the application lifecycle, including originated, denied, withdrawn, incomplete, and approved-but-not-accepted applications. For the purpose of binary classification, we isolated records to strictly two definitive outcomes. **Originated** is mapped to `target_y = 1`, while **Denied** is mapped to `target_y = 0`.

* **Stratified Sampling:** Given the massive scale of the raw dataset, we applied a **1% stratified proportional sampling fraction** strictly within the restricted two-class subset, preserving the class distribution by design. This yielded a final analytical sample of **139,991** records.

### 1.3 Target Definition and Statistical Interpretation

Because the analysis filters HMDA records to two definitive outcomes, the model estimates the conditional probability of origination within the Originated-or-Denied subset:

$$P(\text{Originated} \mid \text{Originated or Denied})$$

It does not estimate the unconditional probability of origination across all HMDA applications. Withdrawn, incomplete, approved-but-not-accepted, and other non-final outcomes are excluded from the binary target construction. Therefore, all subsequent model performance, feature importance, threshold analysis, and fairness interpretation should be read as applying only to applications with a definitive Originated-or-Denied outcome.





## 2. Data Cleaning and Preprocessing

To ensure data integrity and prevent data leakage, the preprocessing pipeline computes all data-driven transformations strictly on the training set, then applies the learned parameters to the test set. This design keeps the feature engineering, imputation, outlier handling, transformation, and encoding steps consistent with the later modeling workflow.

### 2.1 Datatype Formatting and Feature Engineering

To prevent data leakage, we first removed post-application variables that encode or directly follow from the application outcome. These include the action and denial-reason fields (`action_taken`, `denial_reason-1` through `denial_reason-4`, and `purchaser_type`), as well as loan pricing fields that are recorded only for originated loans (`interest_rate`, `rate_spread`, `total_loan_costs`, `origination_charges`, `discount_points`, `lender_credits`, `hoepa_status`, and `total_points_and_fees`). We then enforced strict numeric and string type casting while preserving missingness for later imputation.

Two additional features were engineered to improve modeling structure. First, `activity_year` was transformed into a categorical `covid_phase` variable to capture macroeconomic shifts across the 2019–2024 study period. Second, HMDA's categorical debt-to-income strings were parsed into an ordinal numeric feature (`debt_to_income_ratio_ord`) together with a binary missingness indicator (`dti_missing_flag`).

### 2.2 Column Dropping Rules

To reduce noise, prevent unnecessary dimensionality growth, and improve computational efficiency, features were systematically dropped based on missingness, variance, and cardinality thresholds.

Any column with a missing value rate exceeding **80%** was dropped directly, since imputing such heavily sparse features would introduce substantial artificial signal. Features lacking sufficient variation were also removed because they provide little predictive value. For continuous columns, we dropped variables with standard deviation `<= 0.03`. For categorical columns, features were dropped if a single dominant category accounted for `>= 98%` of the observations.

To prevent sparse matrix explosion during one-hot encoding, a tiered cardinality filter was applied to categorical features. Columns with more than **8 unique values** were dropped to minimize noise and computational burden, while columns with **5–8 unique values** were flagged for manual consolidation, such as bucketing rare categories to improve robustness.

### 2.3 Imputation

Following missingness analysis via heatmap, we applied a tiered imputation strategy. Numeric fields were imputed using `KNNImputer(k=5)` with temporary scaling to support distance-based imputation accuracy, while categorical features received mode imputation.

To maintain interpretability, selected discrete and ordinal features, including `loan_term` and `debt_to_income_ratio_ord`, were explicitly overridden with median values rather than relying only on KNN-based imputation.

### 2.4 Log1p Transformation

To stabilize variance and mitigate right-skewness, we applied a `log1p` transformation after clipping negative values to zero. This transformation was applied unconditionally to heavy-tailed monetary features, including `loan_amount` and `property_value`.

For `income`, the transformation was applied dynamically only when its training distribution showed strong right skewness, defined as a skewness coefficient greater than `1.0`. This avoids unnecessary transformation when the feature distribution does not require it.

### 2.5 Outlier Handling

To mitigate the impact of extreme anomalies, we applied Winsorization by clipping numeric values at the 1st and 99th percentiles. The clipping boundaries were derived strictly from the training set and then applied to both training and test splits.

This process was restricted to high-cardinality numeric features with more than 10 unique values, preserving the integrity of binary and ordinal indicators. DTI-related features were explicitly excluded from Winsorization to avoid distorting their ordinal meaning.

### 2.6 One-Hot Encoding and Final Matrix Construction

We used a `ColumnTransformer` to convert the cleaned feature matrix into a modeling-ready design matrix. Key steps included binarizing `state_code`, treating low-cardinality numeric-coded fields as categorical, and applying `OneHotEncoder` to categorical features. The encoder was configured to handle unseen categories during test-time transformation and to avoid unnecessary collinearity.

After cleaning and imputation, the training matrix used for feature-signal analysis contains **111,992 rows** and **45 predictors**, including **32 numeric** and **13 categorical** features. After one-hot encoding and final matrix construction, the modeling-ready sparse design matrix contains **97 encoded features**.

This final matrix is used as the common handoff point for the downstream EDA, dimensionality-reduction analysis, and supervised modeling stages.





## 3. EDA: Feature Signal and Redundancy Analysis

This stage evaluates which features are most associated with approval outcomes while preserving strict train-only analysis.

All statistics are computed on finalized training artifacts from the preprocessing pipeline (`X_train_imputed` with `y_train`) to ensure consistency with the modeling workflow and avoid leakage from test data.

- Train size: **111,992 rows**
- Predictors analyzed: **45** (`32` numeric, `13` categorical)
- Numeric association: **point-biserial correlation**, with Spearman correlation used as a robustness check
- Categorical association: **Cramer's V**
- Post-OHE category signal: approval-rate **lift** relative to the base training approval rate
- Redundancy diagnostic: numeric **Spearman** correlation (`|rho| >= 0.8`)

### 3.1 Key Signal Features

The numeric ranking is led by tract-level housing and population structure variables, including `tract_one_to_four_family_homes`, `tract_owner_occupied_units`, and `tract_population`, along with loan-structure and valuation variables such as `loan_to_value_ratio` and `lien_status`. Individual raw numeric features show relatively weak marginal association with approval outcomes, but the strongest signals concentrate in meaningful areas — local tract structure, collateral context, and mortgage-structure indicators — which supports the later use of multivariate models where weak individual effects can still contribute through nonlinear combinations and interactions.

<img src="./eda_insights/eda_insights_plots/fig06_numeric_signal_top15.png" alt="Figure 3.1. Top numeric signal features" width="700">

**Figure 3.1.** Top numeric features ranked by absolute point-biserial correlation with `target_y` on the training split.

The categorical ranking is led by loan-product, loan-purpose, geography, and applicant-demographic grouping variables. Although the absolute association values are small, categorical fields can still contribute useful signal after one-hot encoding when individual categories are evaluated at the OHE level, as shown in Section 3.2.

<img src="./eda_insights/eda_insights_plots/fig07_categorical_signal_top15.png" alt="Figure 3.2. Top categorical signal features" width="700">

**Figure 3.2.** Top categorical features ranked by Cramer's V with `target_y` on the training data.

### 3.2 Post-OHE Category Lift

To make the categorical signal more interpretable after one-hot encoding, category-level lift is computed as the difference between the approval rate when a category indicator is active and the overall base approval rate in the training split.

Only categories with sufficient support are emphasized, which prevents rare categories from appearing important only because of very small sample sizes. The strongest positive-lift categories include applications processed through the standard automated underwriting system (`aus_grouped_Standard_AUS`), VA loans (`loan_type_3`), and certain submission and joint-application indicators, all of which show approval rates roughly 6 to 9 percentage points above the base rate. The strongest negative-lift categories are concentrated among specific loan-purpose, observation-status, and demographic indicators that are associated with below-average approval likelihood.

<img src="./eda_insights/eda_insights_plots/fig13_ohe_lift.png" alt="Figure 3.3. OHE category lift" width="700">

**Figure 3.3.** Category-level lift relative to the base approval rate, highlighting the strongest positive and negative OHE effects among categories with support of at least 1%.

This post-OHE view complements the aggregate categorical ranking. Cramer's V identifies which original categorical fields are useful overall, while lift shows which specific encoded category levels drive the strongest positive or negative directional signal.

### 3.3 Multicollinearity / Redundancy Findings

We identified 11 highly correlated numeric pairs (`|Spearman| >= 0.8`). Most of these relationships occur among co-applicant observation variables, applicant observation variables, tract-level housing-count variables, and mortgage-structure flags.

<img src="./eda_insights/eda_insights_plots/fig08_numeric_redundancy_heatmap.png" alt="Figure 3.4. Numeric redundancy heatmap" width="700">

**Figure 3.4.** Spearman correlation structure among numeric training predictors; dense warm/cool clusters indicate redundant feature groups.

This indicates meaningful redundancy in the raw feature space. The redundancy is not necessarily harmful for tree-based models, but it matters for interpretation because correlated predictors can split or share attribution. For this reason, the later modeling stage keeps feature-level interpretability in view rather than treating dimensionality reduction as the only solution.

Overall, the EDA shows that the dataset contains weak but meaningful supervised signal, concentrated in tract structure, loan terms, product/purpose groupings, and selected OHE category levels. It also shows that the feature space is partly redundant, which motivates the TruncatedSVD analysis in Section 4 as a compact structural diagnostic rather than a replacement for feature-level model interpretation.





## 4. Unsupervised Learning (TruncatedSVD)

### 4.1 Method and Controls

Given the finalized one-hot encoding pipeline outputs a sparse design matrix, `TruncatedSVD` is the appropriate dimensionality-reduction method for this project.Unlike dense PCA, SVD operates directly on sparse matrices, preserves computational efficiency, and avoids unnecessary densification risk.

The unsupervised step is fit on training data only, with explicit alignment checks against `y_train`. This keeps the full workflow consistent with train-only preprocessing and prevents leakage from test data.

For interpretability, component loadings are mapped back to encoded feature names (`feat_names`, or encoder-derived names when needed).

### 4.2 Results and Practical Use

SVD component selection is based on cumulative explained variance thresholds (90% / 95%) over a practical component range. In the notebook results, the binary OHE block reaches 90% and 95% explained variance at 21 and 27 components, the standardized continuous block at 8 and 10 components, and the combined sparse input at 22 and 30 components.

This shows that most of the combined feature-space structure can be represented with roughly 22–30 latent components, reducing the original encoded input while preserving the main variance structure.

<img src="./eda_insights/eda_insights_plots/fig09_svd_cumvar_9.png" alt="Figure 4.1. SVD cumulative explained variance for binary OHE columns" width="700">

**Figure 4.1.** Cumulative explained variance when SVD is applied to binary OHE columns only.

<img src="./eda_insights/eda_insights_plots/fig10_svd_cumvar_10.png" alt="Figure 4.2. SVD cumulative explained variance for continuous columns" width="700">

**Figure 4.2.** Cumulative explained variance for standardized continuous columns under SVD.

<img src="./eda_insights/eda_insights_plots/fig11_svd_cumvar_11.png" alt="Figure 4.3. SVD cumulative explained variance for combined sparse input" width="700">

**Figure 4.3.** Cumulative explained variance for the combined sparse input (continuous + OHE).

<img src="./eda_insights/eda_insights_plots/fig12_svd2_embedding.png" alt="Figure 4.4. SVD 2D embedding" width="700">

**Figure 4.4.** Two-dimensional SVD embedding (subsample) colored post-hoc by `target_y` to visualize global separability; target labels are not used when fitting SVD.

The 2D embedding is used as a diagnostic visualization rather than the final modeling representation. Top-loading summaries identify which encoded variables drive each latent component, supporting interpretability despite dimensionality reduction. For example, the first SVD component is dominated by loan-structure and binary indicator variables, while the second component is driven primarily by tract-level housing and population variables, along with related continuous loan and income measures.

### 4.3 Modeling Implication

The SVD analysis confirms that the encoded feature space contains substantial redundancy, with roughly 22–30 latent components capturing most of the variance in the combined input. While this suggests SVD-reduced features could serve as a compact modeling representation, the final modeling stage retains the feature-level OHE matrix to preserve per-feature interpretability required for the SHAP attribution and fairness audit conducted in Section 5 and beyond. Tree-based ensembles are robust to the redundancy identified here, so the loss in compactness is offset by the gain in interpretability.

Feature-signal rankings remain the interpretability anchor, while the SVD results provide supporting evidence that the high-dimensional sparse feature space carries meaningful but compressible variance structure.



## 5. Model Building and Evaluation

### 5.1 Model Selection and Rationale



The modeling strategy is guided by the EDA findings in Sections 3 and 4. Moderate feature-target correlations, nonlinear relationships, correlated numeric predictors, and latent structure in the encoded feature space suggest that denial outcomes are driven by interactions among multiple variables rather than by a single dominant predictor. These patterns support the use of ensemble methods, while Logistic Regression is retained as an interpretable linear baseline.

Three supervised models were evaluated. Logistic Regression provides a transparent baseline suited to the regulatory context of mortgage lending. Random Forest was included as a nonlinear ensemble model that can handle interaction-heavy and partially redundant features, with class imbalance addressed using `class_weight="balanced"`. LightGBM was included as a boosted-tree model for capturing more complex localized interactions in the imbalanced tabular dataset, with imbalance handled using `scale_pos_weight`.





### 5.2 Training Setup




All three models are evaluated under 5-fold stratified cross-validation on a training set of 111,992 observations, with stratification ensuring the 20.4% denial rate is preserved across every fold. PR-AUC is used as the primary scoring metric because the target class is imbalanced and the main modeling objective is to identify denial cases reliably. ROC-AUC, precision, recall, and F1 are retained as secondary diagnostic metrics.



### 5.3 Hyperparameter Tuning


Logistic Regression was tuned using GridSearchCV over penalty type and regularization strength. Random Forest and LightGBM were tuned using RandomizedSearchCV with 20 and 30 iterations respectively, given their larger search spaces. Search ranges and selected values are summarized in Table 5.1.
<div>


| Model | Parameter | Search Range | Best Value |
|---|---|---|---|
| Logistic Regression | `penalty` | L1, L2 | L2 |
| Logistic Regression | `C` | 0.01, 0.1, 1, 10 | 1 |
| Random Forest | `n_estimators` | 300, 500 | 300 |
| Random Forest | `max_depth` | None, 10, 20 | 20 |
| Random Forest | `min_samples_leaf` | 1, 5, 20 | 5 |
| Random Forest | `max_features` | sqrt, 0.3 | 0.3 |
| LightGBM | `num_leaves` | 31, 63, 127 | 63 |
| LightGBM | `learning_rate` | 0.03, 0.05, 0.1 | 0.03 |
| LightGBM | `n_estimators` | 500, 1000 | 500 |
| LightGBM | `min_child_samples` | 20, 100 | 20 |
| LightGBM | `subsample` | 0.8, 1.0 | 1.0 |
| LightGBM | `colsample_bytree` | 0.8, 1.0 | 1.0 |

**Table 5.1.** Hyperparameter search spaces and selected values.

### 5.4 Probability Calibration



Because the LightGBM output is interpreted as a continuous denial risk score rather than only as a binary classification label, probability calibration was applied to make predicted risks better align with observed denial rates. This is important for downstream threshold selection in Section 5.6.4, where classification cutoffs depend on the reliability of the predicted probabilities.

Isotonic regression calibration was applied to the LightGBM model by `CalibratedClassifierCV` with `cv = 5`. Isotonic regression was preferred over Platt scaling because it imposes only a monotonicity constraint on the mapping from raw scores to calibrated probabilities, allowing it to correct nonlinear calibration distortions typical of gradient-boosted models.



### 5.5 Leakage Detection and Remediation



During model interpretation, the `initially_payable_to_institution` feature group appeared as an unexpectedly important predictor, with one dummy variable ranking third in permutation importance. Although leakage screening is ideally completed during data cleaning, this feature was not initially removed because the raw data description did not clearly indicate that the variable was unavailable at the time of application. Its unusually high importance prompted a closer review of the feature’s timing and meaning.

The review showed that the variable represents the institution to which the loan was initially payable, which is determined after loan origination rather than during the application stage. To avoid target leakage, all four `initially_payable_to_institution` dummy variables were removed, reducing the feature set from 97 to 93 predictors. The calibrated LightGBM model was then refit on the corrected feature set. All subsequent evaluation, interpretation, and threshold analysis are based on this post-removal model.



### 5.6 Interpretability Methods



Three complementary methods are used to interpret the final LightGBM pipeline. SHAP values are computed with TreeSHAP on a raw LightGBM model refit using the final 93-feature set and selected hyperparameters, supporting both global feature ranking and local case-level explanations. Permutation importance is calculated on the calibrated model as the drop in PR-AUC after each test-set feature is shuffled. Partial Dependence Plots are then generated for the top features to show their marginal relationships with predicted approval probability; denial risk is interpreted in the opposite direction.



### 5.7 Results

#### 5.7.1 Cross-Validation Performance




Cross-validation results in Table 5.1 show a clear separation between the linear baseline and the two ensemble models during model selection. Logistic Regression achieved a mean ROC-AUC of 0.6911, while Random Forest and LightGBM both exceeded 0.885. This gap supports the EDA finding that mortgage denial outcomes are better captured by nonlinear and interaction-based models than by a purely linear specification. LightGBM achieved the highest mean ROC-AUC and PR-AUC, with the lowest fold-to-fold variance, and was therefore selected as the final model candidate for calibration and refitting.



| Model | ROC-AUC | PR-AUC |
|---|---:|---:|
| Logistic Regression | 0.6911 ± 0.0039 | 0.8628 ± 0.0021 |
| Random Forest | 0.8854 ± 0.0008 | 0.9554 ± 0.0009 |
| LightGBM | 0.8874 ± 0.0006 | 0.9565 ± 0.0007 |

<p><strong>Table 5.1.</strong> Cross-validation performance summary during model selection, reported as mean ± standard deviation across 5 folds.</p>


#### 5.7.2 Leakage-Adjusted Test-Set Performance and Calibration



After removing the leakage features, the calibrated LightGBM model was refit on the corrected 93-feature set and evaluated on the held-out test set. The final model achieved a ROC-AUC of 0.8577 and PR-AUC of 0.9455, indicating strong ranking performance after leakage remediation. The Brier score of 0.0952 is used to assess probability calibration, while the denial-class F1 score of 0.6357 summarizes the precision-recall balance for identifying likely denials. These results are treated as the final deployment-relevant performance estimates.



| Model | Feature Set | ROC-AUC | PR-AUC | Brier | F1<br>(Denial) |
|---|---:|---:|---:|---:|---:|
| Calibrated LightGBM | 93 features | 0.8577 | 0.9455 | 0.0952 | 0.6357 |

**Table 5.2.** Leakage-adjusted final model performance on the held-out test set.

#### 5.7.3 Feature Importance and Interpretability



SHAP values show that the final LightGBM model relies most heavily on `debt_to_income_ratio_ord`, `aus_grouped_Standard_AUS`, `loan_to_value_ratio`, `loan_amount`, and `property_value` (Figure 5.1). These predictors are meaningful in a mortgage approval context. Debt-to-income ratio reflects the applicant’s repayment burden, while loan-to-value ratio captures how much of the property value is financed by the loan. Their high importance suggests that the model’s predictions are driven mainly by affordability and collateral-risk factors. The importance of `aus_grouped_Standard_AUS` also indicates that the automated underwriting system category carries substantial information about the final application outcome.



<img src="./model/results/shap_summary.png" alt="Figure 5.1. SHAP summary plot for the leakage-adjusted LightGBM model" width="550">
<p><strong>Figure 5.1.</strong> SHAP summary plot for the leakage-adjusted LightGBM model.</p>


Partial Dependence Plots further clarify how the top predictors affect predicted approval probability (Figure 5.2). Approval probability drops sharply when `debt_to_income_ratio_ord` reaches high levels, which is consistent with the idea that heavier debt obligations reduce repayment capacity. `loan_to_value_ratio` also shows weaker approval patterns at higher values, reflecting greater risk when the loan amount is large relative to the property value. `loan_amount` and `property_value` have milder marginal effects, suggesting that they provide supporting context rather than acting as dominant decision drivers.




<img src="./model/results/pdp_top5.png" alt="Figure 5.2. Partial dependence plots for the top five features" width="1000">

<p><strong>Figure 5.2.</strong> Partial dependence plots for the top five features.</p>



#### 5.7.4 Threshold Analysis

Threshold analysis translates the final calibrated model probabilities into an operational decision rule, where applications scoring below the threshold are flagged as likely denials for further review. A lower threshold flags fewer applications (only the highest-risk cases), while a higher threshold flags more.

Table 5.4 shows the tradeoff between catching more likely denials and limiting over-flagging. As the threshold increases from 0.3 to 0.7, denial recall (the share of true denial cases the model catches) rises from 0.393 to 0.632, while the flagged rate (the share of all applications sent for review) also increases from 8.8% to 19.0%.

Denial-class F1 is highest at threshold 0.7, with precision of 0.678, recall of 0.632, and a flagged rate of 19.0%. However, threshold 0.6 produces nearly the same F1 score while flagging only 15.6% of applications, making it a more practical default operating point if the goal is to balance risk detection with operational efficiency.


| Threshold | Precision<br>(Denial) | Recall<br>(Denial) | F1<br>(Denial) | Flagged Rate |
|---:|---:|---:|---:|---:|
| 0.3 | 0.911 | 0.393 | 0.550 | 0.088 |
| 0.4 | 0.864 | 0.457 | 0.598 | 0.108 |
| 0.5 | 0.806 | 0.525 | 0.636 | 0.133 |
| 0.6 | 0.753 | 0.576 | 0.653 | 0.156 |
| 0.7 | 0.678 | 0.632 | 0.654 | 0.190 |

<p><strong>Table 5.4.</strong> Threshold sensitivity analysis for the final calibrated LightGBM model.</p>

## 6. Deployment: Mortgage Approval Estimator



The final calibrated LightGBM model was deployed as an interactive R Shiny application: https://crystalguo.shinyapps.io/mortgage-approval-estimator/. The app integrates the trained model, the isotonic calibration layer, and pre-computed TreeSHAP attributions to support real-time scoring and explanation. All model artifacts are derived from the training and test pipeline described in Sections 2 through 5; the app does not query live HMDA data.

The interface is organized into two tabs. **Tab 1 (Score an Application)** lets users enter applicant details such as income, loan amount, and debt-to-income ratio, and returns an approval probability, a plain-language verdict ("Likely approved", "On the fence", or "Likely denied"), and the top three reasons behind the prediction. **Tab 2 (About the Model)** provides a non-technical overview of how the model was built, its overall accuracy, the most influential features, and its known limitations.

Two extensions are planned for future work: a batch input function that allows users to upload a CSV of applications for portfolio-level scoring, and an exploration dashboard for examining prediction distributions and fairness metrics across demographic and geographic subgroups.



## 7. Summary and Limitation

This study built an end-to-end machine learning pipeline to predict mortgage application outcomes using HMDA data, with LightGBM selected as the final calibrated model after comparing against Logistic Regression and Random Forest baselines. The model achieved a test ROC-AUC of 0.890 and PR-AUC of 0.946, with debt-to-income ratio, automated underwriting system, and loan-to-value ratio identified as the strongest predictors of denial.

A primary limitation is that the dataset was restricted to California and Texas to manage data volume. While these are two of the largest U.S. mortgage markets, the model's generalizability to other states with different regional lending patterns, regulatory environments, or demographic compositions has not been tested. Future work could extend the pipeline to additional states and evaluate whether the same feature drivers and threshold tradeoffs hold across more diverse geographic markets.

## 8. Reference

Federal Financial Institutions Examination Council. (2025). Home Mortgage Disclosure 
  Act (HMDA) Data Browser [Data set]. Consumer Financial Protection Bureau. 
  https://ffiec.cfpb.gov/data-browser/

U.S. Census Bureau. (2025). State Population Totals and Components of Change: 2020–2025
  (Vintage 2025) [Data set]. https://www.census.gov/data/tables/time-series/demo/popest/2020s-state-total.html


## 9. Contributions

| Team Member | Contribution |
|---|---|
| Shuxuan Xu (sx2412) | Data preprocessing, including data cleaning and feature engineering. Wrote the corresponding report section. |
| Fangyi Lin (fl2748) | Exploratory data analysis and visualization. Wrote the corresponding report section. |
| Veronica Joe (jj3470) | EDA for modeling, including analysis used to support model selection. Wrote the corresponding report section. |
| Crystal Guo (lg3434) | Model building and evaluation, including model selection and development. Wrote the corresponding report section. |